In [1]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ==========================================
# 1. CONFIGURATION & PATHS
# ==========================================
# Update these paths to match your folder structure
train_dir = '/home/mohamed/autonom_ws/src/Action-Emotions_deep-learning-model/emotions/train'  # e.g., /home/mohamed/dataset/train
test_dir = '/home/mohamed/autonom_ws/src/Action-Emotions_deep-learning-model/emotions/test'    # e.g., /home/mohamed/dataset/test

img_size = (48, 48) # FER-2013 standard size
batch_size = 64     # Number of images to process at once

# ==========================================
# 2. DATA GENERATORS ( The "Pipeline" )
# ==========================================

# A. Training Generator (With Data Augmentation)
# We randomly rotate, flip, and zoom images so the model doesn't just memorize specific pixels.
train_datagen = ImageDataGenerator(
    rescale=1./255,          # Normalize pixels to 0-1 range
    rotation_range=10,       # Rotate image slightly (10 degrees)
    width_shift_range=0.1,   # Shift image horizontally
    height_shift_range=0.1,  # Shift image vertically
    zoom_range=0.1,          # Zoom in/out slightly
    horizontal_flip=True,    # Flip image left-right
    fill_mode='nearest'
)

# B. Validation/Test Generator (No Augmentation)
# We only rescale the test data. We want to test on "real" images, not distorted ones.
test_datagen = ImageDataGenerator(rescale=1./255)

print("Loading Training Data:")
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    color_mode='grayscale',  # IMPORTANT: Your model expects 1 channel
    batch_size=batch_size,
    class_mode='sparse', # Use 'categorical' for 7 emotions
    shuffle=True
)

print("Loading Test/Validation Data:")
validation_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    color_mode='grayscale',
    batch_size=batch_size,
    class_mode='sparse',
    shuffle=False # Don't shuffle validation data so we can check confusion matrix later if needed
)

2025-12-22 01:32:47.525165: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/mohamed/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Loading Training Data:
Found 28709 images belonging to 7 classes.
Loading Test/Validation Data:
Found 7178 images belonging to 7 classes.


# Model Design #

In [2]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization


model = Sequential()

# --- Block 1 ---
# Knowledge Applied: Two Conv layers followed by Pooling (VGG Style)
model.add(Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(48, 48, 1)))
model.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
model.add(BatchNormalization()) # Helps training stabilize
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))        # Prevent overfitting


# --- Block 2 ---
model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

# --- Block 3 ---
model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

# --- Classification Head ---
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))

# Output Layer
# Knowledge Applied: 7 classes = 7 neurons. Softmax for probability distribution.
model.add(Dense(7, activation='softmax')) 

# Summary to check the architecture
model.summary()

/home/mohamed/.local/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1766359971.146730  236183 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 750 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1660 Ti, pci bus id: 0000:01:00.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 48, 48, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 48, 48, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 48, 48, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 24, 24, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 24, 24, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 12, 12, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 12, 12, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         3,591 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,652,775 (10.12 MB)

 Trainable params: 2,651,303 (10.11 MB)

 Non-trainable params: 1,472 (5.75 KB)

## Training the model ##

In [ ]:
import os

# Allow TensorFlow to use the driver's JIT if ptxas is missing
os.environ['XLA_FLAGS'] = '--xla_gpu_unsafe_fallback_to_driver_on_ptxas_not_found=true'

# --- 1. CHANGE IMPORT HERE ---
from tensorflow.keras.optimizers import Adagrad
from tensorflow.keras.callbacks import EarlyStopping

# ==========================================
# 4. COMPILE AND TRAIN (UPDATED FOR SPARSE)
# ==========================================

# 1. Define the Early Stopping Callback
early_stopping = EarlyStopping(
    monitor='val_loss',         # Watch the validation loss
    patience=8,                 # Stop if it doesn't improve for 5 epochs
    min_delta=0.001,            # Minimum change to qualify as an improvement
    restore_best_weights=True,  # IMPORTANT: Revert to the best model found
    verbose=1
)

# 2. Add it to your training loop
model.compile(
    optimizer=Adagrad(learning_rate=0.001), 
    loss='sparse_categorical_crossentropy', 
    metrics=['accuracy']
)

epochs = 150 

print(f"Starting training for {epochs} epochs...")

history = model.fit(
    train_generator,
    epochs=epochs,                 # Set a high max, early stopping will cut it short
    validation_data=validation_generator,
    validation_steps=validation_generator.n // validation_generator.batch_size,
    callbacks=[early_stopping] 
)

# ==========================================
# 5. SAVE THE MODEL
# ==========================================
# Updated filename to reflect the optimizer change
model.save('emotion_model_sparse_adagrad.keras')
print("Model saved as 'emotion_model_sparse_adagrad.keras'")

Starting training for 150 epochs...
Epoch 1/150


2025-12-22 01:32:54.211797: I external/local_xla/xla/service/service.cc:163] XLA service 0x738a58012b30 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-12-22 01:32:54.211811: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce GTX 1660 Ti, Compute Capability 7.5
2025-12-22 01:32:54.274720: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-12-22 01:32:54.633516: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002
2025-12-22 01:32:55.354395: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[64,32,48,48]{3,2,1,0}, u8[0]{0}) custom-call(f32[64,1,48,48]{3,2,1,0}, f32[32,1,3,3]{3,2,1,0}, f32[32]{0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_

  6/449 ━━━━━━━━━━━━━━━━━━━━ 11s 27ms/step - accuracy: 0.1531 - loss: 3.0240

I0000 00:00:1766359980.465927  236318 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


132/449 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.1821 - loss: 2.7208

2025-12-22 01:33:04.974011: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[37,32,48,48]{3,2,1,0}, u8[0]{0}) custom-call(f32[37,1,48,48]{3,2,1,0}, f32[32,1,3,3]{3,2,1,0}, f32[32]{0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]}
2025-12-22 01:33:05.002602: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[37,32,48,48]{3,2,1,0}, u8[0]{0}) custom-call(f32[37,32,48,48]{3,2,1,0}, f32[32,32,3,3]{3,2,1,0}, f32[32]{0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cud

448/449 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.2011 - loss: 2.4303

2025-12-22 01:33:20.545651: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[64,32,48,48]{3,2,1,0}, u8[0]{0}) custom-call(f32[64,1,48,48]{3,2,1,0}, f32[32,1,3,3]{3,2,1,0}, f32[32]{0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kRelu","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]}
2025-12-22 01:33:20.584870: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[64,32,48,48]{3,2,1,0}, u8[0]{0}) custom-call(f32[64,32,48,48]{3,2,1,0}, f32[32,32,3,3]{3,2,1,0}, f32[32]{0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cud

449/449 ━━━━━━━━━━━━━━━━━━━━ 30s 49ms/step - accuracy: 0.2159 - loss: 2.2272 - val_accuracy: 0.2443 - val_loss: 1.9492
Epoch 2/150
449/449 ━━━━━━━━━━━━━━━━━━━━ 17s 38ms/step - accuracy: 0.2336 - loss: 2.0816 - val_accuracy: 0.2909 - val_loss: 1.8849
Epoch 3/150
449/449 ━━━━━━━━━━━━━━━━━━━━ 16s 36ms/step - accuracy: 0.2540 - loss: 1.9963 - val_accuracy: 0.3079 - val_loss: 1.8383
Epoch 4/150
449/449 ━━━━━━━━━━━━━━━━━━━━ 16s 35ms/step - accuracy: 0.2857 - loss: 1.8972 - val_accuracy: 0.3345 - val_loss: 1.7082
Epoch 5/150
449/449 ━━━━━━━━━━━━━━━━━━━━ 16s 36ms/step - accuracy: 0.3215 - loss: 1.7767 - val_accuracy: 0.3456 - val_loss: 1.7871
Epoch 6/150
449/449 ━━━━━━━━━━━━━━━━━━━━ 16s 36ms/step - accuracy: 0.3392 - loss: 1.7138 - val_accuracy: 0.4097 - val_loss: 1.5235
Epoch 7/150
449/449 ━━━━━━━━━━━━━━━━━━━━ 16s 36ms/step - accuracy: 0.3603 - loss: 1.6591 - val_accuracy: 0.4237 - val_loss: 1.4727
Epoch 8/150
449/449 ━━━━━━━━━━━━━━━━━━━━ 17s 38ms/step - accuracy: 0.3791 - loss: 1.6089 - val_

In [ ]:
import matplotlib
# CRITICAL: This line must be BEFORE you import pyplot
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# ==========================================
# 6. VISUALIZATION & LOGGING
# ==========================================

# A. Save the raw numbers to CSV
hist_df = pd.DataFrame(history.history)
hist_df.to_csv('training_history.csv', index=False)
print("Training history saved to 'training_history.csv'")

# B. Plot Accuracy and Loss
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Accuracy
ax1.plot(history.history['accuracy'], label='Train Accuracy')
ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend(loc='lower right')
ax1.grid(True)

# Plot 2: Loss
ax2.plot(history.history['loss'], label='Train Loss')
ax2.plot(history.history['val_loss'], label='Validation Loss')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend(loc='upper right')
ax2.grid(True)

# Save the training graphs
plt.savefig('training_graphs.png', dpi=300)
print("Graphs saved as 'training_graphs.png'")
plt.close() # Close to free up memory

# ==========================================
# 7. CONFUSION MATRIX
# ==========================================
print("\nGenerating Confusion Matrix...")

# 1. Get Predictions
# Important: Reset generator to start from the beginning
validation_generator.reset() 

# Predict on all validation data
preds = model.predict(validation_generator, verbose=1)
y_pred = np.argmax(preds, axis=1) # Convert probabilities to class labels (0, 1, 2...)
y_true = validation_generator.classes # True labels from the folder structure

# 2. Compute Matrix
cm = confusion_matrix(y_true, y_pred)
class_labels = list(validation_generator.class_indices.keys()) # e.g. ['angry', 'happy', ...]

# 3. Plot Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_labels, 
            yticklabels=class_labels)

plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

# 4. Save Confusion Matrix
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300)
print("Confusion Matrix saved as 'confusion_matrix.png'")
plt.close()

# Optional: Print a text report for precision/recall details
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_labels))

Training history saved to 'training_history.csv'
Graphs saved as 'training_graphs.png'

Generating Confusion Matrix...
110/113 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step

2025-12-22 01:41:45.978917: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[10,32,48,48]{3,2,1,0}, u8[0]{0}) custom-call(f32[10,1,48,48]{3,2,1,0}, f32[32,1,3,3]{3,2,1,0}, f32[32]{0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kRelu","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]}
2025-12-22 01:41:46.024993: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[10,32,48,48]{3,2,1,0}, u8[0]{0}) custom-call(f32[10,32,48,48]{3,2,1,0}, f32[32,32,3,3]{3,2,1,0}, f32[32]{0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cud

113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step
Confusion Matrix saved as 'confusion_matrix.png'

Classification Report:
              precision    recall  f1-score   support

       angry       0.51      0.51      0.51       958
     disgust       0.67      0.13      0.21       111
        fear       0.40      0.41      0.41      1024
       happy       0.79      0.84      0.82      1774
     neutral       0.54      0.63      0.58      1233
         sad       0.50      0.38      0.43      1247
    surprise       0.71      0.75      0.73       831

    accuracy                           0.60      7178
   macro avg       0.59      0.52      0.53      7178
weighted avg       0.59      0.60      0.59      7178

